# 01 — EDA & Data Preparation

**Proje:** Credit Risk Scoring (Home Credit Default Risk)

Bu defter, projenin ilk durağı. Burada yalnızca ana eğitim tablosunu (`application_train.csv`) güvenli ve **bellek dostu** şekilde yükleyip ilk sağlık kontrollerini yapıyoruz. Ağır birleştirme (bureau, previous_application vb.) ve özellik mühendisliği sonraki defterlerde gelecek.

**Bu defterin hedefleri**
1. `data/raw/application_train.csv` dosyasını yükle.
2. Sayısal sütunları downcast ederek RAM kullanımını düşür (`float64 → float32`, küçük integer tipleri).
3. Hızlı sağlık kontrolü: `shape`, `dtypes`, hedef değişken (`TARGET`) dağılımı, eksik veri özeti.

In [3]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    load_csv,
    missing_value_report,
    split_feature_types,
    plot_target_distribution,
    plot_categorical_default_rate,
    plot_numeric_by_target,
    RAW_DIR,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DIR}")

Python executable: c:\Users\Dgkn0\cursorprojects\Credit_Risk_Project\.venv\Scripts\python.exe
Python version   : 3.11.6
Project root: c:\Users\Dgkn0\cursorprojects\Credit_Risk_Project
Raw data dir: C:\Users\Dgkn0\cursorprojects\Credit_Risk_Project\data\raw


## 1) `application_train.csv` yükleme

`src.utils.load_csv` fonksiyonu dosyayı `data/raw/` altından okur ve `reduce_mem_usage` ile sayısal tipleri downcast eder. Bu sayede aynı içerik çok daha az RAM ile taşınabilir hale gelir.

In [4]:
app_train = load_csv("application_train.csv")
app_train.head()

Loading: C:\Users\Dgkn0\cursorprojects\Credit_Risk_Project\data\raw\application_train.csv
Shape: (307511, 122)
Memory usage: 536.69 MB -> 378.62 MB (29.5% reduction)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,...,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,...,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,...,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.

## 2) Hızlı sağlık kontrolü

Aşağıdaki üç hücre veriyi tanımak için kritik:
- **Boyut & dtypes:** beklenen sütun sayısı ve indirgenmiş tipler
- **TARGET dağılımı:** sınıf dengesizliğinin (yaklaşık %8) doğrulanması
- **Eksik veri özeti:** ileride doldurma stratejisini şekillendirecek

In [5]:
print(f"Shape: {app_train.shape}")
dtype_counts = app_train.dtypes.astype(str).value_counts()
print("\nDtype counts:")
print(dtype_counts)

Shape: (307511, 122)

Dtype counts:
float32    65
int8       37
str        16
int32       2
int16       2
Name: count, dtype: int64


In [6]:
target_counts = app_train["TARGET"].value_counts().sort_index()
target_ratio = app_train["TARGET"].value_counts(normalize=True).sort_index()

target_summary = pd.DataFrame({"count": target_counts, "ratio": target_ratio})
target_summary.index.name = "TARGET"
target_summary

,count,ratio
TARGET,,
0,282686,0.919271
1,24825,0.080729


In [7]:
missing_value_report(app_train, top=15)

,missing,missing_ratio,dtype
COMMONAREA_AVG,214865,0.698723,float32
COMMONAREA_MODE,214865,0.698723,float32
COMMONAREA_MEDI,214865,0.698723,float32
NONLIVINGAPARTMENTS_MEDI,213514,0.694330,float32
NONLIVINGAPARTMENTS_MODE,213514,0.694330,float32
NONLIVINGAPARTMENTS_AVG,213514,0.694330,float32
FONDKAPREMONT_MODE,210295,0.683862,str
LIVINGAPARTMENTS_AVG,210199,0.683550,float32
LIVINGAPARTMENTS_MEDI,210199,0.683550,float32
LIVINGAPARTMENTS_MODE,210199,0.683550,float32


## 3) `TARGET` dağılımı (görsel)

Sayısal tabloyu üst hücrede gördük. Aynı bilgiyi sınıf dengesizliğini somutlaştırmak için bar grafikle de gösterelim. Pozitif (temerrüt) sınıf payı yaklaşık `%8` — bu oran modelleme aşamasında `class_weight`, `scale_pos_weight` veya kalibrasyon stratejilerini doğrudan etkileyecek.

In [ ]:
overall_default_rate = float(app_train["TARGET"].mean())
print(f"Overall default rate: {overall_default_rate:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
plot_target_distribution(app_train, target="TARGET", ax=ax)
plt.tight_layout()
plt.show()

## 4) Özellik tipleri ayrımı

Modelleme öncesi sütunları üç gruba ayırıyoruz:
- **numeric:** doğrudan model girişi olabilir
- **categorical:** ileride encode edilmesi gereken (≤50 unique)
- **high_cardinality:** stratejisi farklı olacak (target encoding vb.)

`SK_ID_CURR` (kimlik) ve `TARGET` ayrımdan dışlanır.

In [ ]:
feature_groups = split_feature_types(app_train, exclude=["SK_ID_CURR", "TARGET"])

for group, cols in feature_groups.items():
    print(f"{group:>17}: {len(cols)}")

print("\nCategorical columns:")
print(feature_groups["categorical"])

## 5) Kategorik özellikler — TARGET'a göre temerrüt oranı

Aşağıda 4 önemli kategorik değişkenin her bir kategorisindeki **temerrüt oranı** (P(TARGET=1)) gösteriliyor. Kesik çizgi genel ortalama (`%8.07`).

Anlamlı sinyaller:
- `NAME_CONTRACT_TYPE`: nakit krediler genelde daha yüksek temerrütlü
- `CODE_GENDER`: erkek başvurucularda oran biraz daha yüksek
- `NAME_EDUCATION_TYPE`: eğitim seviyesi düştükçe risk artıyor
- `NAME_FAMILY_STATUS`: medeni hal segmentasyonu

In [ ]:
cat_focus = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, cat_focus):
    plot_categorical_default_rate(
        app_train,
        col,
        target="TARGET",
        overall_rate=overall_default_rate,
        ax=ax,
    )
plt.tight_layout()
plt.show()

## 6) Kritik numerik özellikler — TARGET'a göre dağılım

`EXT_SOURCE_1/2/3` Home Credit yarışmasının en güçlü sayısal sinyalleri (dış kredi skor benzeri, 0–1 aralığında). `DAYS_BIRTH` ise negatif gün cinsinden olduğu için `AGE_YEARS = -DAYS_BIRTH / 365.25` dönüşümünü yapıyoruz. Eğri ayrılığı ne kadar belirginse, değişken o kadar bilgi verici.

In [ ]:
app_train["AGE_YEARS"] = (-app_train["DAYS_BIRTH"] / 365.25).astype("float32")

num_focus = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "AGE_YEARS"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.flat, num_focus):
    plot_numeric_by_target(app_train, col, target="TARGET", ax=ax)
plt.tight_layout()
plt.show()